# Rough Trapezium Signed-Distance Map

The smooth trapezium is used as a support curve. On each side, a non-negative sinusoidal roughness is subtracted along the inward normal, so the wall is displaced outward. The number of waves is rounded side by side so each side contains an integer number of complete waves. The roughness is tapered to zero near the corners.


In [ ]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt

try:
    from numba import njit
except Exception:
    def njit(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        def decorator(func):
            return func
        return decorator

FIGURE_DIR = Path("Figures")
FIGURE_DIR.mkdir(exist_ok=True)


In [ ]:
TOP = 2.3
BOTTOM = 4.0
HEIGHT = 2.7

ROUGHNESS_AMPLITUDE = 0.08       # maximum outward displacement
TARGET_FREQUENCY = 2.0           # target waves per unit length
TAPER_FRACTION = 0.12            # fraction of each side where roughness fades to zero
INVERSE_N_PER_SIDE = 900         # sampled segments per side for the inverse map
INNER_SCALE = 0.15              # inner trapezium used by the four injective wall patches


In [ ]:
@njit(fastmath=True)
def _clamp(v, lo, hi):
    if v < lo:
        return lo
    if v > hi:
        return hi
    return v


@njit(fastmath=True)
def _dot2(x, y):
    return x * x + y * y


@njit(fastmath=True)
def _perimeter(TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    side = math.sqrt((r1 - r2) * (r1 - r2) + HEIGHT * HEIGHT)
    return BOTTOM + TOP + 2.0 * side


@njit(fastmath=True)
def _side_length(TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    return math.sqrt((r1 - r2) * (r1 - r2) + HEIGHT * HEIGHT)


@njit(fastmath=True)
def _edge_data(edge_id, TOP, BOTTOM, HEIGHT):
    r1 = 0.5 * BOTTOM
    r2 = 0.5 * TOP
    he = 0.5 * HEIGHT
    side = _side_length(TOP, BOTTOM, HEIGHT)

    if edge_id == 0:
        vx = -r1
        vy = -he
        tx = 1.0
        ty = 0.0
        length = BOTTOM
    elif edge_id == 1:
        vx = r1
        vy = -he
        tx = (r2 - r1) / side
        ty = HEIGHT / side
        length = side
    elif edge_id == 2:
        vx = r2
        vy = he
        tx = -1.0
        ty = 0.0
        length = TOP
    else:
        vx = -r2
        vy = he
        tx = (-r1 + r2) / side
        ty = -HEIGHT / side
        length = side

    nx = -ty
    ny = tx
    return vx, vy, tx, ty, nx, ny, length


@njit(fastmath=True)
def _edge_from_s(s, TOP, BOTTOM, HEIGHT):
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    side = _side_length(TOP, BOTTOM, HEIGHT)

    s_mod = s - math.floor(s)
    S = s_mod * perimeter

    if S < BOTTOM:
        return 0, S
    S -= BOTTOM
    if S < side:
        return 1, S
    S -= side
    if S < TOP:
        return 2, S
    S -= TOP
    return 3, S


@njit(fastmath=True)
def _interior_angle(vertex_id, TOP, BOTTOM, HEIGHT):
    prev_edge = vertex_id - 1
    if prev_edge < 0:
        prev_edge = 3
    next_edge = vertex_id

    _, _, tx_prev, ty_prev, _, _, _ = _edge_data(prev_edge, TOP, BOTTOM, HEIGHT)
    _, _, tx_next, ty_next, _, _, _ = _edge_data(next_edge, TOP, BOTTOM, HEIGHT)

    ux = -tx_prev
    uy = -ty_prev
    vx = tx_next
    vy = ty_next
    c = _clamp(ux * vx + uy * vy, -1.0, 1.0)
    return math.acos(c)


@njit(fastmath=True)
def _edge_s_interval(edge_id, d, TOP, BOTTOM, HEIGHT):
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    side = _side_length(TOP, BOTTOM, HEIGHT)

    if edge_id == 0:
        S0 = 0.0
        length = BOTTOM
    elif edge_id == 1:
        S0 = BOTTOM
        length = side
    elif edge_id == 2:
        S0 = BOTTOM + side
        length = TOP
    else:
        S0 = BOTTOM + side + TOP
        length = side

    q = abs(d)
    start_vertex = edge_id
    end_vertex = edge_id + 1
    if end_vertex == 4:
        end_vertex = 0

    a0 = _interior_angle(start_vertex, TOP, BOTTOM, HEIGHT)
    a1 = _interior_angle(end_vertex, TOP, BOTTOM, HEIGHT)

    m0 = 0.0
    m1 = 0.0
    if q > 0.0:
        m0 = q / math.tan(0.5 * a0)
        m1 = q / math.tan(0.5 * a1)

    if m0 + m1 >= length:
        mid = S0 + 0.5 * length
        return mid / perimeter, mid / perimeter

    return (S0 + m0) / perimeter, (S0 + length - m1) / perimeter


@njit(fastmath=True)
def _edge_wave_count(length, target_frequency):
    n = int(math.floor(target_frequency * length + 0.5))
    if n < 1:
        n = 1
    return n


@njit(fastmath=True)
def _smoothstep_with_derivatives(z):
    if z <= 0.0:
        return 0.0, 0.0, 0.0
    if z >= 1.0:
        return 1.0, 0.0, 0.0
    f = z * z * (3.0 - 2.0 * z)
    fp = 6.0 * z - 6.0 * z * z
    fpp = 6.0 - 12.0 * z
    return f, fp, fpp


@njit(fastmath=True)
def _taper_with_derivatives(u, taper_fraction):
    taper = _clamp(taper_fraction, 1.0e-6, 0.499)

    left, left_z, left_zz = _smoothstep_with_derivatives(u / taper)
    dl = left_z / taper
    ddl = left_zz / (taper * taper)

    right, right_z, right_zz = _smoothstep_with_derivatives((1.0 - u) / taper)
    dr = -right_z / taper
    ddr = right_zz / (taper * taper)

    envelope = left * right
    envelope_u = dl * right + left * dr
    envelope_uu = ddl * right + 2.0 * dl * dr + left * ddr
    return envelope, envelope_u, envelope_uu


@njit(fastmath=True)
def _roughness_profile(sigma, length, amplitude, target_frequency, taper_fraction):
    u = _clamp(sigma / length, 0.0, 1.0)
    n_waves = _edge_wave_count(length, target_frequency)
    phase = 2.0 * math.pi * n_waves * u

    wave = 0.5 * (1.0 - math.cos(phase))
    wave_u = math.pi * n_waves * math.sin(phase)
    wave_uu = 2.0 * math.pi * math.pi * n_waves * n_waves * math.cos(phase)

    envelope, envelope_u, envelope_uu = _taper_with_derivatives(u, taper_fraction)

    h = amplitude * envelope * wave
    h_u = amplitude * (envelope_u * wave + envelope * wave_u)
    h_uu = amplitude * (envelope_uu * wave + 2.0 * envelope_u * wave_u + envelope * wave_uu)

    hp = h_u / length
    hpp = h_uu / (length * length)
    return h, hp, hpp


@njit(fastmath=True)
def trapetium_border(s, TOP, BOTTOM, HEIGHT):
    edge_id, sigma = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    vx, vy, tx, ty, _, _, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = _clamp(sigma, 0.0, length)
    return vx + sigma * tx, vy + sigma * ty


@njit(fastmath=True)
def trapetium_rough_border(s, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    edge_id, sigma = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    vx, vy, tx, ty, nx, ny, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = _clamp(sigma, 0.0, length)
    h, _, _ = _roughness_profile(sigma, length, amplitude, target_frequency, taper_fraction)
    px = vx + sigma * tx
    py = vy + sigma * ty
    return px - h * nx, py - h * ny


@njit(fastmath=True)
def trapetium_rough_normal(s, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    edge_id, sigma = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    _, _, tx, ty, nx, ny, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = _clamp(sigma, 0.0, length)
    _, hp, _ = _roughness_profile(sigma, length, amplitude, target_frequency, taper_fraction)
    speed = math.sqrt(1.0 + hp * hp)

    # Rough inward normal. If hp = 0, this is the original side normal.
    nr_t = hp / speed
    nr_n = 1.0 / speed
    nr_x = nr_t * tx + nr_n * nx
    nr_y = nr_t * ty + nr_n * ny
    return nr_x, nr_y


@njit(fastmath=True)
def trapetium_rough_map(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    bx, by = trapetium_rough_border(s, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    nx, ny = trapetium_rough_normal(s, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    return bx + d * nx, by + d * ny


@njit(fastmath=True)
def trapetium_rough_jacobian(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    edge_id, sigma = _edge_from_s(s, TOP, BOTTOM, HEIGHT)
    _, _, tx, ty, nx, ny, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = _clamp(sigma, 0.0, length)
    _, hp, hpp = _roughness_profile(sigma, length, amplitude, target_frequency, taper_fraction)
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)

    speed = math.sqrt(1.0 + hp * hp)
    speed3 = speed * speed * speed

    nt = hp / speed
    nn = 1.0 / speed
    j01 = nt * tx + nn * nx
    j11 = nt * ty + nn * ny

    nprime_t = hpp / speed3
    nprime_n = -hp * hpp / speed3

    a_t = 1.0 + d * nprime_t
    a_n = -hp + d * nprime_n

    j00 = perimeter * (a_t * tx + a_n * nx)
    j10 = perimeter * (a_t * ty + a_n * ny)
    return j00, j01, j10, j11


@njit(fastmath=True)
def trapetium_rough_jacobian_det(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    j00, j01, j10, j11 = trapetium_rough_jacobian(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    return j00 * j11 - j01 * j10


@njit(fastmath=True)
def trapetium_rough_jacobian_inverse(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    j00, j01, j10, j11 = trapetium_rough_jacobian(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    det = j00 * j11 - j01 * j10
    return j11 / det, -j01 / det, -j10 / det, j00 / det


@njit(fastmath=True)
def trapetium_rough_metric(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    j00, j01, j10, j11 = trapetium_rough_jacobian(s, d, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    g00 = j00 * j00 + j10 * j10
    g01 = j00 * j01 + j10 * j11
    g11 = j01 * j01 + j11 * j11
    return g00, g01, g11


In [ ]:
def build_rough_boundary_table(n_per_side, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    side = _side_length(TOP, BOTTOM, HEIGHT)
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    lengths = [BOTTOM, side, TOP, side]

    s_values = []
    S0 = 0.0
    for length in lengths:
        local = np.linspace(0.0, length, n_per_side, endpoint=False)
        s_values.extend((S0 + local) / perimeter)
        S0 += length
    s_values.append(1.0)

    s_values = np.asarray(s_values, dtype=np.float64)
    xy = np.array([
        trapetium_rough_border(s, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
        for s in s_values
    ])
    return s_values, xy[:, 0], xy[:, 1]


@njit(fastmath=True)
def _point_inside_polygon(x, y, x_poly, y_poly):
    inside = False
    n = x_poly.shape[0]
    j = n - 1
    for i in range(n):
        yi = y_poly[i]
        yj = y_poly[j]
        xi = x_poly[i]
        xj = x_poly[j]
        if ((yi > y) != (yj > y)):
            x_cross = (xj - xi) * (y - yi) / (yj - yi + 1.0e-300) + xi
            if x < x_cross:
                inside = not inside
        j = i
    return inside


@njit(fastmath=True)
def trapetium_rough_map_inverse(
    x,
    y,
    s_table,
    x_table,
    y_table,
    TOP,
    BOTTOM,
    HEIGHT,
    amplitude,
    target_frequency,
    taper_fraction,
):
    best_d2 = 1.0e300
    best_s = 0.0

    nseg = s_table.shape[0] - 1
    for i in range(nseg):
        ax = x_table[i]
        ay = y_table[i]
        bx = x_table[i + 1]
        by = y_table[i + 1]
        ex = bx - ax
        ey = by - ay
        denom = ex * ex + ey * ey
        u = 0.0
        if denom > 0.0:
            u = ((x - ax) * ex + (y - ay) * ey) / denom
            u = _clamp(u, 0.0, 1.0)
        cx = ax + u * ex
        cy = ay + u * ey
        dx = x - cx
        dy = y - cy
        d2 = dx * dx + dy * dy
        if d2 < best_d2:
            best_d2 = d2
            best_s = s_table[i] + u * (s_table[i + 1] - s_table[i])

    if best_s >= 1.0:
        best_s -= 1.0

    d = math.sqrt(best_d2)
    if not _point_inside_polygon(x, y, x_table, y_table):
        d = -d

    return best_s, d


s_table, x_table, y_table = build_rough_boundary_table(
    INVERSE_N_PER_SIDE,
    TOP,
    BOTTOM,
    HEIGHT,
    ROUGHNESS_AMPLITUDE,
    TARGET_FREQUENCY,
    TAPER_FRACTION,
)


## Plot The Rough Boundary


In [ ]:
s_plot = np.linspace(0.0, 1.0, 1600)
smooth = np.array([trapetium_border(s, TOP, BOTTOM, HEIGHT) for s in s_plot])
rough = np.array([
    trapetium_rough_border(s, TOP, BOTTOM, HEIGHT, ROUGHNESS_AMPLITUDE, TARGET_FREQUENCY, TAPER_FRACTION)
    for s in s_plot
])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(smooth[:, 0], smooth[:, 1], color="0.65", lw=1.0, label="trapetium support")
ax.plot(rough[:, 0], rough[:, 1], color="black", lw=1.0, label="rough wall")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("rough trapezium boundary")
ax.grid(True)
ax.legend()
fig.savefig(FIGURE_DIR / "rough_trapetium_boundary.png", dpi=300, bbox_inches="tight")
plt.show()


## Plot The Multi-Patch Forward Map


In [ ]:
@njit(fastmath=True)
def _edge_start_S(edge_id, TOP, BOTTOM, HEIGHT):
    side = _side_length(TOP, BOTTOM, HEIGHT)
    if edge_id == 0:
        return 0.0
    if edge_id == 1:
        return BOTTOM
    if edge_id == 2:
        return BOTTOM + side
    return BOTTOM + side + TOP


@njit(fastmath=True)
def _s_from_edge_u(edge_id, u, TOP, BOTTOM, HEIGHT):
    perimeter = _perimeter(TOP, BOTTOM, HEIGHT)
    _, _, _, _, _, _, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    S = _edge_start_S(edge_id, TOP, BOTTOM, HEIGHT) + _clamp(u, 0.0, 1.0) * length
    s = S / perimeter
    if s >= 1.0:
        s -= 1.0
    return s


@njit(fastmath=True)
def rough_patch_wall(edge_id, u, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction):
    s = _s_from_edge_u(edge_id, u, TOP, BOTTOM, HEIGHT)
    return trapetium_rough_border(s, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)


@njit(fastmath=True)
def rough_patch_inner(edge_id, u, TOP, BOTTOM, HEIGHT, inner_scale):
    s = _s_from_edge_u(edge_id, u, TOP, BOTTOM, HEIGHT)
    x, y = trapetium_border(s, TOP, BOTTOM, HEIGHT)
    return inner_scale * x, inner_scale * y


@njit(fastmath=True)
def rough_patch_map(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale):
    u = _clamp(u, 0.0, 1.0)
    v = _clamp(v, 0.0, 1.0)
    bx, by = rough_patch_wall(edge_id, u, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    cx, cy = rough_patch_inner(edge_id, u, TOP, BOTTOM, HEIGHT, inner_scale)
    return (1.0 - v) * bx + v * cx, (1.0 - v) * by + v * cy


@njit(fastmath=True)
def rough_patch_jacobian(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale):
    u = _clamp(u, 0.0, 1.0)
    v = _clamp(v, 0.0, 1.0)
    _, _, tx, ty, nx, ny, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    sigma = u * length
    _, hp, _ = _roughness_profile(sigma, length, amplitude, target_frequency, taper_fraction)

    bx, by = rough_patch_wall(edge_id, u, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    cx, cy = rough_patch_inner(edge_id, u, TOP, BOTTOM, HEIGHT, inner_scale)

    bux = length * (tx - hp * nx)
    buy = length * (ty - hp * ny)
    cux = inner_scale * length * tx
    cuy = inner_scale * length * ty

    j00 = (1.0 - v) * bux + v * cux
    j10 = (1.0 - v) * buy + v * cuy
    j01 = cx - bx
    j11 = cy - by
    return j00, j01, j10, j11


@njit(fastmath=True)
def rough_patch_jacobian_det(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale):
    j00, j01, j10, j11 = rough_patch_jacobian(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale)
    return j00 * j11 - j01 * j10


@njit(fastmath=True)
def rough_patch_jacobian_inverse(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale):
    j00, j01, j10, j11 = rough_patch_jacobian(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale)
    det = j00 * j11 - j01 * j10
    return j11 / det, -j01 / det, -j10 / det, j00 / det


@njit(fastmath=True)
def rough_patch_metric(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale):
    j00, j01, j10, j11 = rough_patch_jacobian(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale)
    g00 = j00 * j00 + j10 * j10
    g01 = j00 * j01 + j10 * j11
    g11 = j01 * j01 + j11 * j11
    return g00, g01, g11


@njit(fastmath=True)
def rough_patch_map_inverse(
    x,
    y,
    s_table,
    x_table,
    y_table,
    TOP,
    BOTTOM,
    HEIGHT,
    amplitude,
    target_frequency,
    taper_fraction,
    inner_scale,
):
    s_near, signed_d = trapetium_rough_map_inverse(
        x,
        y,
        s_table,
        x_table,
        y_table,
        TOP,
        BOTTOM,
        HEIGHT,
        amplitude,
        target_frequency,
        taper_fraction,
    )

    edge_id, sigma = _edge_from_s(s_near, TOP, BOTTOM, HEIGHT)
    _, _, _, _, _, _, length = _edge_data(edge_id, TOP, BOTTOM, HEIGHT)
    u = _clamp(sigma / length, 0.0, 1.0)

    bx, by = rough_patch_wall(edge_id, u, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction)
    cx, cy = rough_patch_inner(edge_id, u, TOP, BOTTOM, HEIGHT, inner_scale)
    vx = cx - bx
    vy = cy - by
    vv = vx * vx + vy * vy
    v = 0.0
    if vv > 0.0:
        v = ((x - bx) * vx + (y - by) * vy) / vv
    v = _clamp(v, 0.0, 1.0)

    for _ in range(20):
        mx, my = rough_patch_map(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale)
        fx = mx - x
        fy = my - y
        j00, j01, j10, j11 = rough_patch_jacobian(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale)
        det = j00 * j11 - j01 * j10
        if abs(det) < 1.0e-12:
            break
        du = (j11 * fx - j01 * fy) / det
        dv = (-j10 * fx + j00 * fy) / det
        u = _clamp(u - du, 0.0, 1.0)
        v = _clamp(v - dv, 0.0, 1.0)
        if du * du + dv * dv < 1.0e-22:
            break

    mx, my = rough_patch_map(edge_id, u, v, TOP, BOTTOM, HEIGHT, amplitude, target_frequency, taper_fraction, inner_scale)
    error = math.sqrt((mx - x) * (mx - x) + (my - y) * (my - y))
    return edge_id, u, v, signed_d, error


fig, ax = plt.subplots(figsize=(6, 4.5))

u_plot = np.linspace(0.0, 1.0, 500)
v_values = [0.0, 0.10, 0.25, 0.50, 0.75, 1.0]
for v in v_values:
    color = None
    for edge_id in range(4):
        xy = np.array([
            rough_patch_map(edge_id, u, v, TOP, BOTTOM, HEIGHT, ROUGHNESS_AMPLITUDE, TARGET_FREQUENCY, TAPER_FRACTION, INNER_SCALE)
            for u in u_plot
        ])
        label = f"v = {v:.2f}" if edge_id == 0 else None
        line, = ax.plot(xy[:, 0], xy[:, 1], lw=1.0, label=label, color=color)
        color = line.get_color()

for edge_id in range(4):
    for u in np.linspace(0.0, 1.0, 9):
        xy = np.array([
            rough_patch_map(edge_id, u, v, TOP, BOTTOM, HEIGHT, ROUGHNESS_AMPLITUDE, TARGET_FREQUENCY, TAPER_FRACTION, INNER_SCALE)
            for v in np.linspace(0.0, 1.0, 120)
        ])
        ax.plot(xy[:, 0], xy[:, 1], color="0.75", lw=0.5)

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("multi-patch rough map: wall at v = 0")
ax.grid(True)
ax.legend()
fig.savefig(FIGURE_DIR / "rough_trapetium_patch_map.png", dpi=300, bbox_inches="tight")
plt.show()


## Plot The Inverse Map Distance


In [ ]:
pad = 0.25
x_min = x_table.min() - pad
x_max = x_table.max() + pad
y_min = y_table.min() - pad
y_max = y_table.max() + pad

xg = np.linspace(x_min, x_max, 150)
yg = np.linspace(y_min, y_max, 150)
Xg, Yg = np.meshgrid(xg, yg)
Dg = np.empty_like(Xg)

for i in range(Xg.shape[0]):
    for j in range(Xg.shape[1]):
        _, Dg[i, j] = trapetium_rough_map_inverse(
            Xg[i, j],
            Yg[i, j],
            s_table,
            x_table,
            y_table,
            TOP,
            BOTTOM,
            HEIGHT,
            ROUGHNESS_AMPLITUDE,
            TARGET_FREQUENCY,
            TAPER_FRACTION,
        )

fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.contourf(Xg, Yg, Dg, levels=40, cmap="coolwarm")
ax.plot(x_table, y_table, color="black", lw=1.0)
ax.contour(Xg, Yg, Dg, levels=[0.0], colors="black", linewidths=1.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("inverse map distance d(x,y)")
fig.colorbar(im, ax=ax, label="d")
fig.savefig(FIGURE_DIR / "rough_trapetium_inverse_distance.png", dpi=300, bbox_inches="tight")
plt.show()


## Plot And Check The Jacobian And Metric


In [ ]:
u_grid = np.linspace(0.0, 1.0, 220)
v_grid = np.linspace(0.0, 1.0, 160)
Ug, Vg = np.meshgrid(u_grid, v_grid)

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
axes = axes.ravel()
min_abs_det = np.inf

for edge_id, ax in enumerate(axes):
    DET_J = np.empty_like(Ug)
    for i in range(Ug.shape[0]):
        for j in range(Ug.shape[1]):
            DET_J[i, j] = rough_patch_jacobian_det(
                edge_id,
                Ug[i, j],
                Vg[i, j],
                TOP,
                BOTTOM,
                HEIGHT,
                ROUGHNESS_AMPLITUDE,
                TARGET_FREQUENCY,
                TAPER_FRACTION,
                INNER_SCALE,
            )
    min_abs_det = min(min_abs_det, np.min(np.abs(DET_J)))
    im = ax.contourf(Ug, Vg, DET_J, levels=35)
    ax.set_title(f"wall patch {edge_id}")
    ax.set_xlabel("u")
    ax.set_ylabel("v")
    fig.colorbar(im, ax=ax, label="det J")

fig.suptitle("multi-patch Jacobian determinant")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "rough_trapetium_patch_jacobian.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"min |det J| over wall patches = {min_abs_det:.6e}")


## Create Particles Inside The Rough Trapezium


In [ ]:
particle_radius = 0.05
N = 200
safety = 1.2
max_trials = 200000
particle_seed = 2
inverse_tolerance = 1.0e-6


def minimum_neighbour_distance(x):
    if len(x) < 2:
        return np.inf, np.full(len(x), np.inf)

    d_min = np.inf
    nearest = np.full(len(x), np.inf)
    for i in range(len(x)):
        d2 = np.sum((x - x[i])**2, axis=1)
        d2[i] = np.inf
        nearest[i] = np.sqrt(np.min(d2))
        d_min = min(d_min, nearest[i])
    return d_min, nearest


def generate_rough_patch_particles_no_overlap(
    N,
    particle_radius,
    safety=1.1,
    max_trials=200000,
    seed=1,
    inverse_tolerance=1.0e-6,
):
    rng = np.random.default_rng(seed)

    min_dist = safety * 2.0 * particle_radius
    min_dist2 = min_dist * min_dist

    x_min = x_table.min() + particle_radius
    x_max = x_table.max() - particle_radius
    y_min = y_table.min() + particle_radius
    y_max = y_table.max() - particle_radius

    q_acc = np.empty((N, 3), dtype=np.float64)
    x_acc = np.empty((N, 2), dtype=np.float64)

    count = 0
    trials = 0

    while count < N and trials < max_trials:
        trials += 1

        x_new = rng.uniform(x_min, x_max)
        y_new = rng.uniform(y_min, y_max)

        edge_id, u_new, v_new, signed_d, error = rough_patch_map_inverse(
            x_new,
            y_new,
            s_table,
            x_table,
            y_table,
            TOP,
            BOTTOM,
            HEIGHT,
            ROUGHNESS_AMPLITUDE,
            TARGET_FREQUENCY,
            TAPER_FRACTION,
            INNER_SCALE,
        )

        if signed_d <= particle_radius:
            continue
        if error > inverse_tolerance:
            continue

        if count == 0:
            accept = True
        else:
            d2 = np.sum((x_acc[:count] - np.array([x_new, y_new]))**2, axis=1)
            accept = np.min(d2) > min_dist2

        if accept:
            q_acc[count, 0] = edge_id
            q_acc[count, 1] = u_new
            q_acc[count, 2] = v_new
            x_acc[count, 0] = x_new
            x_acc[count, 1] = y_new
            count += 1

    q_acc = q_acc[:count]
    x_acc = x_acc[:count]
    d_min, nearest = minimum_neighbour_distance(x_acc)

    info = {
        "requested": N,
        "placed": count,
        "trials": trials,
        "particle_diameter": 2.0 * particle_radius,
        "minimum_allowed_distance": min_dist,
        "minimum_actual_distance": d_min,
    }
    return q_acc, x_acc, nearest, info


q_particles, x_particles, nearest, particle_info = generate_rough_patch_particles_no_overlap(
    N=N,
    particle_radius=particle_radius,
    safety=safety,
    max_trials=max_trials,
    seed=particle_seed,
    inverse_tolerance=inverse_tolerance,
)

q0 = q_particles.astype(np.float32)
p0 = np.zeros((q0.shape[0], 2), dtype=np.float32)
x0 = x_particles.astype(np.float32)

print(particle_info)

fig, (ax_phys, ax_q) = plt.subplots(1, 2, figsize=(10, 4))

ax_phys.plot(x_table, y_table, color="black", lw=1.0)
ax_phys.scatter(x_particles[:, 0], x_particles[:, 1], s=40)
ax_phys.set_aspect("equal", adjustable="box")
ax_phys.set_xlabel("x")
ax_phys.set_ylabel("y")
ax_phys.set_title("physical space")
ax_phys.grid(True)

scatter = ax_q.scatter(
    q_particles[:, 1],
    q_particles[:, 2],
    s=1.0,
    c=q_particles[:, 0],
    cmap="tab10",
    vmin=0,
    vmax=4,
)
ax_q.axhline(0.0, color="black", lw=1.0)
ax_q.set_xlim(0.0, 1.0)
ax_q.set_ylim(0.0, 1.0)
ax_q.set_xlabel("u")
ax_q.set_ylabel("v")
ax_q.set_title("multi-patch coordinates")
fig.colorbar(scatter, ax=ax_q, label="wall id")
ax_q.grid(True)

fig.savefig(FIGURE_DIR / "rough_trapetium_patch_particles.png", dpi=300, bbox_inches="tight")
plt.show()


## Notes

The rough boundary is defined by moving each straight side opposite to its inward normal. The roughness amplitude is non-negative, so the wall is displaced outward and the internal volume is not reduced by the wave. The taper removes the wave near the corners.

The normal-offset map is not used as the main forward map because normals from a rough wall can cross. The notebook therefore defines a four-patch map. In each patch, `v = 0` is the rough wall and `v = 1` is a small inner trapezium. The inverse first uses the signed-distance nearest-wall map to choose the wall patch, then solves the patch coordinates `(u,v)` by Newton iteration.

This gives an injective wall-coordinate map for the region between the rough wall and the inner trapezium. The small central trapezium is left as a separate region; it can later be handled with Cartesian coordinates or with another simple central map.


In [ ]:
# Hybrid rough-wall solver run

import time
import importlib
import hybrid_solver as hs

#importlib.reload(hs)

# The hybrid solver starts from physical positions and velocities.
N_sim = x0.shape[0]

v0 = np.zeros_like(x0, dtype=np.float32)
m = np.ones(N_sim, dtype=np.float32)
rad = np.full(N_sim, particle_radius, dtype=np.float32)
group_mobile = np.array([0, N_sim], dtype=np.int64)

xA = np.array([0.0, 0.3], dtype=np.float32)
omega = 8.0 * np.pi

margin = np.linalg.norm(xA) + 4.0 * particle_radius + ROUGHNESS_AMPLITUDE

box = np.array([
    x_table.min() - margin,
    x_table.max() + margin,
    y_table.min() - margin,
    y_table.max() + margin,
], dtype=np.float32)

dt = 1.0e-4
T_max = 5.0
save_every = 50

k_contact = 1.0e5
gamma_contact = 30.0

k_w = k_contact
gamma_w = gamma_contact

g = np.array([0.0, -9.81], dtype=np.float32)

# Warm-up run for Numba compilation.
_ = hs.simulate_hybrid_particles(
    box=box,
    x0=x0,
    v0=v0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=dt,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    TOP=TOP,
    BOTTOM=BOTTOM,
    HEIGHT=HEIGHT,
    ROUGHNESS_AMPLITUDE=ROUGHNESS_AMPLITUDE,
    TARGET_FREQUENCY=TARGET_FREQUENCY,
    TAPER_FRACTION=TAPER_FRACTION,
    INNER_SCALE=INNER_SCALE,
    save_every=99999,
)

# Real run.
t0 = time.perf_counter()

t_h, q_h, p_h, x_h, v_h, mode_h = hs.simulate_hybrid_particles(
    box=box,
    x0=x0,
    v0=v0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=T_max,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    TOP=TOP,
    BOTTOM=BOTTOM,
    HEIGHT=HEIGHT,
    ROUGHNESS_AMPLITUDE=ROUGHNESS_AMPLITUDE,
    TARGET_FREQUENCY=TARGET_FREQUENCY,
    TAPER_FRACTION=TAPER_FRACTION,
    INNER_SCALE=INNER_SCALE,
    save_every=save_every,
    enter_factor=2.0,
    exit_factor=2.5,
    corner_factor=2.0,
)

elapsed = time.perf_counter() - t0
n_steps = int(T_max / dt + 0.5)

print("runtime [s]:", elapsed)
print("steps:", n_steps)
print("particles:", N_sim)
print("time per step [ms]:", 1000.0 * elapsed / n_steps)
print("particle-steps per second:", n_steps * N_sim / elapsed)

print("saved frames:", len(t_h))
print("final time:", t_h[-1])
print("x shape:", x_h.shape)
print("q shape:", q_h.shape)
print("mode shape:", mode_h.shape)

In [ ]:
import pre_post_processing_v16 as pp

ani_phys = pp.animate_rough_physical_simulation(
    t_h,
    x_h,
    x_table,
    y_table,
    TOP=TOP,
    BOTTOM=BOTTOM,
    HEIGHT=HEIGHT,
    xA=xA,
    omega=omega,
    switch_radius=particle_radius,
    enter_factor=2.0,
    stride=2,
    s=40,
    show_trace0=False,
)

plt.close(ani_phys._fig)

pp.to_jshtml(ani_phys, embed_limit_mb=100)

In [ ]:
ani_q = pp.animate_hybrid_coordinate_simulation(
    t_h,
    q_h,
    mode_h,
    TOP=TOP,
    BOTTOM=BOTTOM,
    HEIGHT=HEIGHT,
    ROUGHNESS_AMPLITUDE=ROUGHNESS_AMPLITUDE,
    TARGET_FREQUENCY=TARGET_FREQUENCY,
    TAPER_FRACTION=TAPER_FRACTION,
    INNER_SCALE=INNER_SCALE,
    switch_radius=particle_radius,
    enter_factor=2.0,
    stride=2,
    s=40,
    show_trace0=False,
)

plt.close(ani_q._fig)

pp.to_jshtml(ani_q, embed_limit_mb=50)

In [ ]:
# Last-frame diagnostic: physical space and hybrid coordinates

frame = 300
switch_distance = 2.0 * particle_radius

fig, (ax_phys, ax_q) = plt.subplots(1, 2, figsize=(12, 4.8))

shift = xA * np.sin(omega * t_h[frame])

# -------------------------
# Physical space
# -------------------------

ax_phys.plot(
    x_table + shift[0],
    y_table + shift[1],
    color="black",
    lw=1.0,
    label="rough wall",
)

for edge_id in range(4):
    vx, vy, tx, ty, nx, ny, length = hs._edge_data(
        edge_id, TOP, BOTTOM, HEIGHT
    )

    u0, u1 = pp._hybrid_offset_u_interval(
        hs, edge_id, switch_distance, TOP, BOTTOM, HEIGHT
    )

    xs = []
    ys = []

    for u in np.linspace(u0, u1, 200):
        sigma = u * length
        xs.append(vx + sigma * tx + switch_distance * nx + shift[0])
        ys.append(vy + sigma * ty + switch_distance * ny + shift[1])

    ax_phys.plot(
        xs,
        ys,
        color="red",
        lw=1.2,
        ls=":",
        label="mode threshold" if edge_id == 0 else None,
    )

ax_phys.scatter(
    x_h[frame, :, 0],
    x_h[frame, :, 1],
    s=50,
)

ax_phys.set_aspect("equal", adjustable="box")
ax_phys.set_xlabel("x")
ax_phys.set_ylabel("y")
ax_phys.set_title(f"physical space, t = {t_h[frame]:.3f}")
ax_phys.grid(True)
ax_phys.legend(loc="lower right")


# -------------------------
# Hybrid coordinates
# -------------------------

q_plot = np.full((q_h.shape[1], 2), np.nan)

for i in range(q_h.shape[1]):
    if mode_h[frame, i] == 1:
        edge_id = int(q_h[frame, i, 0])
        u = q_h[frame, i, 1]
        v = q_h[frame, i, 2]
    else:
        x_i = q_h[frame, i, 1]
        y_i = q_h[frame, i, 2]

        edge_id, _, _ = hs.straight_trapezium_nearest_edge(
            x_i, y_i, TOP, BOTTOM, HEIGHT
        )

        u, v = hs.rough_patch_inverse_edge(
            x_i, y_i, edge_id,
            TOP, BOTTOM, HEIGHT,
            ROUGHNESS_AMPLITUDE,
            TARGET_FREQUENCY,
            TAPER_FRACTION,
            INNER_SCALE,
        )

    q_plot[i, 0] = edge_id + u
    q_plot[i, 1] = v

for edge_id in range(5):
    ax_q.axvline(edge_id, color="0.75", lw=0.6)

ax_q.axhline(
    0.0,
    color="black",
    lw=0.8,
    ls="--",
    label="rough wall",
)

for edge_id in range(4):
    vx, vy, tx, ty, nx, ny, length = hs._edge_data(
        edge_id, TOP, BOTTOM, HEIGHT
    )

    u0, u1 = pp._hybrid_offset_u_interval(
        hs, edge_id, switch_distance, TOP, BOTTOM, HEIGHT
    )

    xs = []
    ys = []

    for u_straight in np.linspace(u0, u1, 200):
        sigma = u_straight * length
        x_sep = vx + sigma * tx + switch_distance * nx
        y_sep = vy + sigma * ty + switch_distance * ny

        u_sep, v_sep = hs.rough_patch_inverse_edge(
            x_sep, y_sep, edge_id,
            TOP, BOTTOM, HEIGHT,
            ROUGHNESS_AMPLITUDE,
            TARGET_FREQUENCY,
            TAPER_FRACTION,
            INNER_SCALE,
        )

        xs.append(edge_id + u_sep)
        ys.append(v_sep)

    ax_q.plot(
        xs,
        ys,
        color="red",
        lw=1.2,
        ls=":",
        label="mode threshold" if edge_id == 0 else None,
    )

ax_q.scatter(q_plot[:, 0], q_plot[:, 1], s=20)

ax_q.set_xlim(-0.05, 4.05)
ax_q.set_ylim(-0.02, max(1.0, 1.05 * np.nanmax(q_plot[:, 1])))
ax_q.set_xlabel("wall patch")
ax_q.set_ylabel("v")
ax_q.set_title("hybrid wall-patch coordinates")
ax_q.grid(True)
ax_q.legend(loc="lower right")

plt.tight_layout()
plt.show()

fig.savefig(FIGURE_DIR / "simulation.png", dpi=300, bbox_inches="tight")
plt.show()
